In [ ]:
import os
import pandas as pd

curr_dir = os.getcwd()
# print(curr_dir)
input_csv = f"{curr_dir}/dataset/UCBERKELEY_AMY_6MM_24Mar2026.csv"
output_dir = f"{curr_dir}/dataset/"
os.makedirs(output_dir, exist_ok=True)

PT_ID_COL = "PTID"
DATE_COL = "SCANDATE"

df = pd.read_csv(input_csv)

df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

mri_cols = [c for c in df.columns if "_VOLUME" in c.upper()]
pet_cols = [c for c in df.columns if "_SUVR" in c.upper()]

print(f"MRI ROIs: {len(mri_cols)}")
print(f"PET ROIs: {len(pet_cols)}")

def base_name(col, suffix):
    return col.upper().replace(suffix, "")

mri_map = {base_name(c, "_VOLUME"): c for c in mri_cols}
pet_map = {base_name(c, "_SUVR"): c for c in pet_cols}

common_rois = sorted(set(mri_map.keys()).intersection(set(pet_map.keys())))
print(f"Matched ROIs: {len(common_rois)}")

mri_final = [mri_map[r] for r in common_rois]
pet_final = [pet_map[r] for r in common_rois]

df = df.sort_values([PT_ID_COL, DATE_COL])
df = df.groupby(PT_ID_COL, as_index=False).first()

def filter_sparse(df, cols, threshold=0.7):
    valid_ratio = df[cols].notna().mean(axis=1)
    before = len(df)
    df = df[valid_ratio >= threshold].copy()
    after = len(df)
    print(f"Filtered sparse subjects: {before - after} removed, {after} kept")
    return df

mri_df = df[[PT_ID_COL, DATE_COL] + mri_final].copy()
pet_df = df[[PT_ID_COL, DATE_COL] + pet_final].copy()

mri_df = filter_sparse(mri_df, mri_final, threshold=0.7)
pet_df = filter_sparse(pet_df, pet_final, threshold=0.7)

common_ids = set(mri_df[PT_ID_COL]).intersection(set(pet_df[PT_ID_COL]))
print(f"Common subjects after filtering: {len(common_ids)}")

mri_df = mri_df[mri_df[PT_ID_COL].isin(common_ids)].copy()
pet_df = pet_df[pet_df[PT_ID_COL].isin(common_ids)].copy()

mri_df = mri_df.sort_values(PT_ID_COL).reset_index(drop=True)
pet_df = pet_df.sort_values(PT_ID_COL).reset_index(drop=True)

assert (mri_df[PT_ID_COL].values == pet_df[PT_ID_COL].values).all(), "TID mismatch after alignment!"

assert len(mri_final) == len(pet_final), "ROI count mismatch!"

assert set([base_name(c, "_VOLUME") for c in mri_final]) == \
       set([base_name(c, "_SUVR") for c in pet_final]), \
       "ROI semantic mismatch!"

mri_save = os.path.join(output_dir, "MRI_roi.csv")
pet_save = os.path.join(output_dir, "PET_roi.csv")

mri_df.to_csv(mri_save, index=False)
pet_df.to_csv(pet_save, index=False)

print("Saved MRI ROI:", mri_save)
print("Saved PET ROI:", pet_save)
print("Subjects:", len(mri_df))
print("ROI pairs:", len(common_rois))

/home/shreyamm/Desktop/UPenn/projects/ESE6740
MRI ROIs: 162
PET ROIs: 163
Matched ROIs: 162
Filtered sparse subjects: 15 removed, 2114 kept
Filtered sparse subjects: 15 removed, 2114 kept
Common subjects after filtering: 2114


/tmp/ipykernel_1837922/1872924537.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")


Saved MRI ROI: /home/shreyamm/Desktop/UPenn/projects/ESE6740/dataset/MRI_roi.csv
Saved PET ROI: /home/shreyamm/Desktop/UPenn/projects/ESE6740/dataset/PET_roi.csv
Subjects: 2114
ROI pairs: 162


In [ ]:
import os
import pandas as pd

curr_dir = os.getcwd()
# print(curr_dir)

MRI_path = f"{curr_dir}/dataset/MRI_roi.csv"
PET_path = f"{curr_dir}/dataset/PET_roi.csv"
df_MRI = pd.read_csv(MRI_path)
df_PET = pd.read_csv(PET_path)

In [6]:
df_MRI.head()

,PTID,SCANDATE,ACCUMBENS_AREA_VOLUME,AMYGDALA_VOLUME,BRAINSTEM_VOLUME,CAUDATE_VOLUME,CC_ANTERIOR_VOLUME,CC_CENTRAL_VOLUME,CC_MID_ANTERIOR_VOLUME,CC_MID_POSTERIOR_VOLUME,...,RIGHT_VENTRALDC_VOLUME,RIGHT_VESSEL_VOLUME,SUMMARY_VOLUME,THALAMUS_PROPER_VOLUME,VENTRALDC_VOLUME,VENTRICLE_3RD_VOLUME,VENTRICLE_4TH_VOLUME,VESSEL_VOLUME,WHOLECEREBELLUM_VOLUME,WM_HYPOINTENSITIES_VOLUME
0,002_S_0295,2011-06-10,804.0,3274.0,21090.0,7080.0,933.0,573.0,448.0,737.0,...,4409.0,35.0,283651.0,14748.0,8466.0,2258.0,1487.0,97.0,134606.0,3447.0
1,002_S_0413,2011-06-20,866.0,3081.0,19161.0,6985.0,926.0,458.0,459.0,511.0,...,3747.0,2.0,302323.0,12639.0,7442.0,2558.0,1548.0,18.0,115227.0,2326.0
2,002_S_0685,2010-07-20,668.0,2429.0,20058.0,7058.0,999.0,470.0,474.0,440.0,...,3775.0,63.0,239143.0,12580.0,7411.0,3325.0,1743.0,89.0,122216.0,24250.0
3,002_S_0729,2010-07-30,664.0,1322.0,17015.0,6255.0,834.0,454.0,523.0,593.0,...,2993.0,32.0,231610.0,12019.0,6175.0,1262.0,1954.0,39.0,108853.0,3809.0
4,002_S_1155,2011-01-11,1031.0,2732.0,20991.0,7244.0,1063.0,442.0,489.0,516.0,...,3604.0,92.0,290115.0,14159.0,7492.0,1380.0,1572.0,157.0,127635.0,2582.0


In [7]:
df_PET.head()

,PTID,SCANDATE,ACCUMBENS_AREA_SUVR,AMYGDALA_SUVR,BRAINSTEM_SUVR,CAUDATE_SUVR,CC_ANTERIOR_SUVR,CC_CENTRAL_SUVR,CC_MID_ANTERIOR_SUVR,CC_MID_POSTERIOR_SUVR,...,RIGHT_VENTRALDC_SUVR,RIGHT_VESSEL_SUVR,SUMMARY_SUVR,THALAMUS_PROPER_SUVR,VENTRALDC_SUVR,VENTRICLE_3RD_SUVR,VENTRICLE_4TH_SUVR,VESSEL_SUVR,WHOLECEREBELLUM_SUVR,WM_HYPOINTENSITIES_SUVR
0,002_S_0295,2011-06-10,1.304,1.232,1.637,1.117,1.429,1.454,1.258,1.220,...,1.611,1.587,1.516,1.326,1.551,0.652,1.019,1.454,1.0,1.373
1,002_S_0413,2011-06-20,0.888,0.876,1.514,0.878,1.354,1.266,1.248,1.113,...,1.339,1.105,0.985,1.039,1.331,0.519,0.964,1.028,1.0,1.283
2,002_S_0685,2010-07-20,0.761,0.866,1.438,0.874,1.329,1.471,1.433,1.531,...,1.334,1.072,1.001,1.013,1.318,0.517,0.989,1.040,1.0,1.401
3,002_S_0729,2010-07-30,1.617,1.208,1.490,1.466,1.687,1.562,1.652,1.559,...,1.506,1.578,1.446,1.307,1.470,0.787,0.873,1.576,1.0,1.566
4,002_S_1155,2011-01-11,0.793,0.894,1.335,0.792,1.146,1.051,1.079,1.058,...,1.230,0.949,0.926,1.057,1.253,0.652,0.845,0.965,1.0,0.991


In [ ]:
for col